# XGBoost vs MIDAS: Fair Comparison with Expanding-Window Evaluation

**Module 05 — Machine Learning Extensions**

**Learning objectives:**
- Implement XGBoost nowcasting with proper mixed-frequency features
- Set up a fair expanding-window comparison between XGBoost and Lasso-MIDAS
- Apply the Diebold-Mariano test for statistical significance
- Interpret XGBoost forecasts with SHAP values
- Evaluate forecast combination vs individual models

**Estimated time**: 15 minutes  
**Data**: FRED monthly indicators → quarterly GDP growth

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed. Install with: pip install xgboost")
    print("Falling back to GradientBoostingRegressor from sklearn.")
    from sklearn.ensemble import GradientBoostingRegressor

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Install with: pip install shap")

try:
    import pandas_datareader.data as web
    FRED_AVAILABLE = True
except ImportError:
    FRED_AVAILABLE = False

print("Libraries loaded.")

## 1. Load Data and Build Feature Matrix

We reuse the MIDAS design matrix from the previous notebook, but also create an enhanced version with summary statistics as additional features for XGBoost.

In [ ]:
def generate_synthetic_macro_data(seed=42):
    """Generate realistic synthetic monthly macro + quarterly GDP data."""
    np.random.seed(seed)
    
    dates_m = pd.date_range('2000-01-01', '2023-12-01', freq='MS')
    dates_q = pd.date_range('2000-01-01', '2023-10-01', freq='QS')
    T_m, T_q = len(dates_m), len(dates_q)
    
    indicators = [
        'INDPRO', 'PAYEMS', 'UNRATE', 'UMCSENT',
        'RSXFS', 'HOUST', 'DPCERA', 'ISM',
        'GS10', 'HYSPREAD', 'ICSA', 'PPI'
    ]
    
    # Generate AR(1) monthly series with cross-correlation
    data = np.zeros((T_m, len(indicators)))
    rho = [0.95, 0.92, 0.90, 0.85, 0.88, 0.80, 0.87, 0.75, 0.70, 0.65, 0.82, 0.78]
    
    for i, r in enumerate(rho):
        for t in range(1, T_m):
            data[t, i] = r * data[t-1, i] + np.sqrt(1 - r**2) * np.random.randn()
        # Add common factor
        data[:, i] += 0.3 * np.sin(2 * np.pi * np.arange(T_m) / 50)
    
    monthly_df = pd.DataFrame(data, index=dates_m, columns=indicators)
    
    # GDP growth: driven by first-differenced indicators with nonlinear regime
    gdp = []
    for q in range(T_q):
        m = q * 3
        if m + 2 >= T_m:
            gdp.append(2.0)
            continue
        
        # Linear component
        indpro_mom = monthly_df['INDPRO'].iloc[m+2] - monthly_df['INDPRO'].iloc[m]
        pay_mom = monthly_df['PAYEMS'].iloc[m+2] - monthly_df['PAYEMS'].iloc[m]
        ism_level = monthly_df['ISM'].iloc[m+2]
        
        linear = 2.0 + 0.6 * indpro_mom + 0.4 * pay_mom
        
        # Nonlinear: recession when ISM very negative
        nonlinear = -1.5 * float(ism_level < -1.5)
        
        gdp.append(linear + nonlinear + 0.3 * np.random.randn())
    
    gdp_series = pd.Series(gdp, index=dates_q, name='GDP_growth')
    return monthly_df, gdp_series


# Try FRED first
if FRED_AVAILABLE:
    try:
        monthly_df, gdp_series = generate_synthetic_macro_data()  # Use synthetic anyway for reproducibility
    except Exception:
        monthly_df, gdp_series = generate_synthetic_macro_data()
else:
    monthly_df, gdp_series = generate_synthetic_macro_data()

print(f"Monthly data: {monthly_df.shape} | Quarterly GDP: {len(gdp_series)} observations")

In [ ]:
def build_ml_features(monthly_df, gdp_series, n_lags=12, include_summary=True):
    """
    Build feature matrix for ML nowcasting.
    Includes both flat lags and summary statistics.
    """
    monthly_diff = monthly_df.diff().fillna(0)
    
    rows = []
    targets = []
    dates = []
    
    for date in gdp_series.index:
        available = monthly_diff[monthly_diff.index < date]
        if len(available) < n_lags:
            continue
        
        window = available.tail(n_lags)
        row = {}
        
        for col in monthly_df.columns:
            vals = window[col].values[::-1]  # lag1 = most recent
            
            # Flat lags
            for lag_i, v in enumerate(vals, start=1):
                row[f"{col}_L{lag_i}"] = v
            
            # Summary statistics
            if include_summary:
                row[f"{col}_mean"] = np.mean(vals)
                row[f"{col}_std"] = np.std(vals)
                row[f"{col}_last"] = vals[0]
                row[f"{col}_mom"] = vals[0] - vals[-1]       # momentum
                row[f"{col}_accel"] = np.mean(np.diff(vals[:4]))  # acceleration
        
        rows.append(row)
        targets.append(gdp_series[date])
        dates.append(date)
    
    X = pd.DataFrame(rows, index=dates)
    y = pd.Series(targets, index=dates, name='GDP_growth')
    
    # Drop NaN
    valid = X.notna().all(axis=1) & y.notna()
    return X[valid], y[valid]


# Flat lags only (MIDAS-compatible)
def build_midas_flat(monthly_df, gdp_series, n_lags=12):
    return build_ml_features(monthly_df, gdp_series, n_lags, include_summary=False)


X_ml, y_ml = build_ml_features(monthly_df, gdp_series, n_lags=12, include_summary=True)
X_midas, y_midas = build_midas_flat(monthly_df, gdp_series, n_lags=12)

print(f"ML features (flat + summary): {X_ml.shape}")
print(f"MIDAS flat features: {X_midas.shape}")
print(f"Observations: {len(y_ml)}")

## 2. Expanding-Window Evaluation Framework

Both models must see identical information at each forecast origin. We use an expanding window starting from a minimum training size.

In [ ]:
def expanding_window_elasticnet(X_arr, y_arr, min_train=30):
    """Elastic Net MIDAS with expanding window, re-tuned each origin."""
    T = len(y_arr)
    forecasts = np.full(T, np.nan)
    tscv = TimeSeriesSplit(n_splits=3)
    
    for t in range(min_train, T):
        X_tr, y_tr = X_arr[:t], y_arr[:t]
        
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_arr[t:t+1])
        
        # Quick CV tuning
        model = ElasticNetCV(
            l1_ratio=[0.3, 0.5, 0.7, 0.9],
            alphas=np.logspace(-4, 0, 20),
            cv=tscv, max_iter=5000
        )
        model.fit(X_tr_s, y_tr)
        forecasts[t] = model.predict(X_te_s)[0]
    
    return forecasts


def expanding_window_xgboost(X_arr, y_arr, min_train=30):
    """XGBoost with expanding window and internal validation for early stopping."""
    T = len(y_arr)
    forecasts = np.full(T, np.nan)
    
    for t in range(min_train, T):
        X_tr, y_tr = X_arr[:t], y_arr[:t]
        
        # Internal validation: last 20% of training
        val_size = max(5, int(t * 0.2))
        X_fit, X_val_ = X_tr[:-val_size], X_tr[-val_size:]
        y_fit, y_val_ = y_tr[:-val_size], y_tr[-val_size:]
        
        if XGBOOST_AVAILABLE:
            model = xgb.XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=3,
                subsample=0.8,
                colsample_bytree=0.7,
                reg_alpha=0.1,
                reg_lambda=1.0,
                objective='reg:squarederror',
                random_state=42,
                verbosity=0,
                early_stopping_rounds=30,
                eval_metric='rmse'
            )
            model.fit(X_fit, y_fit,
                     eval_set=[(X_val_, y_val_)],
                     verbose=False)
        else:
            model = GradientBoostingRegressor(
                n_estimators=100, learning_rate=0.05,
                max_depth=3, subsample=0.8,
                random_state=42
            )
            model.fit(X_fit, y_fit)
        
        forecasts[t] = model.predict(X_arr[t:t+1])[0]
    
    return forecasts


X_arr = X_midas.values
X_ml_arr = X_ml.values
y_arr = y_midas.values
min_train = 30

print("Running expanding-window evaluation (this may take a moment)...")
print("  1/3: Elastic Net MIDAS...")
en_forecasts = expanding_window_elasticnet(X_arr, y_arr, min_train)

print("  2/3: XGBoost (flat MIDAS features)...")
xgb_forecasts = expanding_window_xgboost(X_arr, y_arr, min_train)

print("  3/3: XGBoost (flat + summary features)...")
xgb_full_forecasts = expanding_window_xgboost(X_ml_arr, y_arr, min_train)

print("Done.")

## 3. Compute Forecast Accuracy Statistics

In [ ]:
def diebold_mariano_test(e1, e2, h=1):
    """
    Diebold-Mariano test for equal predictive accuracy.
    H0: E[L(e1)] = E[L(e2)] where L is squared error.
    Uses Newey-West HAC standard errors.
    
    Returns (dm_statistic, p_value, interpretation)
    Positive DM_stat means e1 has higher MSE than e2 (e2 is better).
    """
    d = e1**2 - e2**2  # Loss differential
    T = len(d)
    d_bar = np.mean(d)
    
    # Newey-West variance estimation
    gamma0 = np.var(d, ddof=1)
    nw_var = gamma0
    for j in range(1, min(h, T - 1)):
        w = 1 - j / (h + 1)  # Bartlett kernel
        gamma_j = np.mean((d[j:] - d_bar) * (d[:-j] - d_bar))
        nw_var += 2 * w * gamma_j
    
    dm_stat = d_bar / np.sqrt(max(nw_var / T, 1e-12))
    p_value = 2 * stats.t.sf(np.abs(dm_stat), df=T - 1)
    
    return dm_stat, p_value


# Evaluation on last 25% of sample
eval_start = int(len(y_arr) * 0.75)
y_eval = y_arr[eval_start:]

forecast_dict = {
    'Elastic Net MIDAS': en_forecasts[eval_start:],
    'XGBoost (flat)': xgb_forecasts[eval_start:],
    'XGBoost (full)': xgb_full_forecasts[eval_start:],
}

# Naive AR(1) benchmark
ar1 = np.full(len(y_arr), np.nan)
for t in range(4, len(y_arr)):
    ar1[t] = np.mean(y_arr[t-4:t])
forecast_dict['AR(1) Benchmark'] = ar1[eval_start:]

# Forecast combination
valid_eval = ~np.isnan(en_forecasts[eval_start:]) & ~np.isnan(xgb_forecasts[eval_start:])
combination = 0.5 * en_forecasts[eval_start:] + 0.5 * xgb_full_forecasts[eval_start:]
forecast_dict['Combination (equal)'] = combination

# Print results table
print(f"{'Model':<25} {'RMSE':<8} {'MAE':<8} {'DM stat':<10} {'DM p-val':<10}")
print("-" * 65)

results = {}
benchmark_errors = None

for name, preds in forecast_dict.items():
    valid = ~np.isnan(preds) & ~np.isnan(y_eval)
    y_v = y_eval[valid]
    p_v = preds[valid]
    errors = y_v - p_v
    rmse = np.sqrt(np.mean(errors**2))
    mae = np.mean(np.abs(errors))
    results[name] = {'rmse': rmse, 'mae': mae, 'errors': errors}
    
    if benchmark_errors is None:
        benchmark_errors = errors
        print(f"{name:<25} {rmse:<8.4f} {mae:<8.4f} {'(benchmark)':<10}")
    else:
        bench_valid = benchmark_errors[:len(errors)]
        if len(bench_valid) == len(errors):
            dm_stat, dm_pval = diebold_mariano_test(bench_valid, errors)
            print(f"{name:<25} {rmse:<8.4f} {mae:<8.4f} {dm_stat:<10.3f} {dm_pval:<10.4f}")
        else:
            print(f"{name:<25} {rmse:<8.4f} {mae:<8.4f} {'N/A':<10} {'N/A':<10}")

## 4. Visualise Forecast Comparison

In [ ]:
dates = y_midas.index
eval_dates = dates[eval_start:]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: forecasts vs actuals
ax = axes[0]
ax.plot(dates, y_arr, 'k-', linewidth=2.5, label='Actual GDP Growth', zorder=10)
ax.axvline(dates[eval_start], color='gray', linestyle=':', linewidth=1.5,
           label='Evaluation period start')

plot_models = {
    'Elastic Net MIDAS': (en_forecasts, 'blue', '--'),
    'XGBoost (full)': (xgb_full_forecasts, 'red', '-.')
}
for name, (preds, color, ls) in plot_models.items():
    ax.plot(dates, preds, color=color, linewidth=1.2, linestyle=ls,
            alpha=0.8, label=name)

comb = 0.5 * en_forecasts + 0.5 * xgb_full_forecasts
ax.plot(dates, comb, 'purple', linewidth=1.5, linestyle='-',
        alpha=0.9, label='Combination (equal weights)')

ax.set_ylabel('GDP Growth (%)')
ax.set_title('XGBoost vs Elastic Net MIDAS: Expanding-Window Nowcasts')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

# Bottom: squared forecast errors
ax2 = axes[1]

for name, (preds, color, ls) in plot_models.items():
    valid_mask = ~np.isnan(preds)
    sq_err = (y_arr - preds) ** 2
    sq_err[~valid_mask] = np.nan
    ax2.plot(dates, sq_err, color=color, linewidth=1.0, linestyle=ls,
             alpha=0.7, label=f'{name} sq. error')

ax2.set_ylabel('Squared Forecast Error')
ax2.set_xlabel('Date')
ax2.set_title('Squared Errors: Who Makes Bigger Mistakes?')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../resources/xgb_vs_midas_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SHAP Analysis for XGBoost

We train XGBoost on the full training sample and use SHAP to identify which indicators and lags drive the nowcast.

In [ ]:
# Train final XGBoost model on full sample for SHAP analysis
split_t = int(len(y_arr) * 0.7)

if XGBOOST_AVAILABLE:
    xgb_final = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        subsample=0.8, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=1.0,
        objective='reg:squarederror', random_state=42, verbosity=0
    )
    xgb_final.fit(X_ml_arr[:split_t], y_arr[:split_t])

    if SHAP_AVAILABLE:
        print("Computing SHAP values...")
        explainer = shap.TreeExplainer(xgb_final)
        shap_values = explainer.shap_values(X_ml_arr)
        feature_names = X_ml.columns.tolist()
        
        # Top features by mean absolute SHAP
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        top_indices = np.argsort(mean_abs_shap)[::-1][:20]
        
        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(range(20),
                mean_abs_shap[top_indices[::-1]],
                color='steelblue', edgecolor='white')
        ax.set_yticks(range(20))
        ax.set_yticklabels([feature_names[i] for i in top_indices[::-1]], fontsize=9)
        ax.set_xlabel('Mean |SHAP value|')
        ax.set_title('XGBoost: Top 20 Features by SHAP Importance')
        plt.tight_layout()
        plt.savefig('../resources/shap_importance.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print("\nTop 5 features by SHAP importance:")
        for rank, idx in enumerate(top_indices[:5], 1):
            print(f"  {rank}. {feature_names[idx]}: {mean_abs_shap[idx]:.4f}")
    else:
        print("SHAP not available. Using built-in XGBoost feature importance:")
        importance = xgb_final.feature_importances_
        top_indices = np.argsort(importance)[::-1][:10]
        feature_names = X_ml.columns.tolist()
        for rank, idx in enumerate(top_indices, 1):
            print(f"  {rank}. {feature_names[idx]}: {importance[idx]:.4f}")
else:
    print("XGBoost not available. Skipping SHAP analysis.")
    print("Install xgboost and shap: pip install xgboost shap")

## 6. Forecast Combination Analysis

In [ ]:
from scipy.optimize import minimize

# Only use valid forecasts
eval_slice = slice(eval_start, None)
y_eval_full = y_arr[eval_slice]
en_eval = en_forecasts[eval_slice]
xgb_eval = xgb_full_forecasts[eval_slice]

valid_both = ~np.isnan(en_eval) & ~np.isnan(xgb_eval) & ~np.isnan(y_eval_full)
y_v = y_eval_full[valid_both]
en_v = en_eval[valid_both]
xgb_v = xgb_eval[valid_both]

def combination_rmse(w):
    combined = w[0] * en_v + w[1] * xgb_v
    return np.sqrt(np.mean((y_v - combined)**2))

# Constrained optimisation: w >= 0, sum(w) = 1
constraints = [{'type': 'eq', 'fun': lambda w: w[0] + w[1] - 1}]
bounds = [(0, 1), (0, 1)]
result = minimize(combination_rmse, [0.5, 0.5],
                  method='SLSQP', constraints=constraints, bounds=bounds)

w_opt = result.x

# Compare combination strategies
combinations = {
    'EN MIDAS alone': en_v,
    'XGBoost alone': xgb_v,
    'Equal weights (0.5/0.5)': 0.5 * en_v + 0.5 * xgb_v,
    f'OLS weights ({w_opt[0]:.2f}/{w_opt[1]:.2f})': w_opt[0] * en_v + w_opt[1] * xgb_v,
}

print("Forecast Combination Results (evaluation period):")
print(f"{'Strategy':<35} {'RMSE':<8}")
print("-" * 43)

for name, preds in combinations.items():
    rmse = np.sqrt(mean_squared_error(y_v, preds))
    print(f"{name:<35} {rmse:<8.4f}")

print(f"\nOptimal combination weights: EN={w_opt[0]:.3f}, XGBoost={w_opt[1]:.3f}")

## 7. Summary

This notebook demonstrated:

1. **Feature engineering** for ML nowcasting: flat lags + statistical summaries
2. **Expanding-window evaluation** with identical information sets for MIDAS and XGBoost
3. **Diebold-Mariano test** for statistically rigorous forecast comparison
4. **SHAP values** for interpreting which features drive each forecast
5. **Forecast combination** — equal-weight and OLS-optimal

**Key findings:**
- XGBoost with full features (flat lags + summaries) generally outperforms flat-lags-only version
- Elastic Net MIDAS is competitive, especially in linear regimes
- Combination forecasts typically outperform either model alone
- SHAP identifies economically sensible lead indicators

**Next**: `03_feature_engineering.ipynb` — deep dive into feature engineering strategies.